In [2]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from keras import layers
import os
from tensorboard.plugins.hparams import api as hp
from tensorboard import version
from sklearn.model_selection import train_test_split



c:\Users\reziv\Documents\GitHub\Lung_Cancer_Git\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [3]:
df = pd.read_excel('..\\..\\Datasets\\NewDataset\\BlackLineHospitalLungCancerDataset.xlsx', header=0) 

X = df.drop(['Severity'], axis=1)
y = df['Severity']

In [4]:
nums = []
TuningTrials = 3

def run(hparams, num, i): #run_dir,
    #with tf.summary.create_file_writer(run_dir).as_default():
        hp.hparams(hparams)
        accuracy = train_test_model(hparams)
        print(accuracy)
        if (i == 0):
            nums.append(accuracy)
        else: 
            nums[num] += accuracy
        if (i == TuningTrials-1):
            average = nums[num]/TuningTrials
            nums[num] = average
            print(average)
            #tf.summary.scalar(METRIC_ACCURACY, average, step=1)

In [11]:
def train_test_model(hparams):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(hparams[HP_NUM_UNITS], activation=tf.nn.relu),
        tf.keras.layers.Dropout(hparams[HP_DROPOUT]),
        tf.keras.layers.Dense(hparams[HP_NUM_UNITS], activation=tf.nn.relu),
        tf.keras.layers.Dropout(hparams[HP_DROPOUT]),
        tf.keras.layers.Dense(hparams[HP_NUM_UNITS], activation=tf.nn.relu),
        tf.keras.layers.Dropout(hparams[HP_DROPOUT]),
        tf.keras.layers.Dense(hparams[HP_NUM_UNITS], activation=tf.nn.relu),
        tf.keras.layers.Dropout(hparams[HP_DROPOUT]),
        tf.keras.layers.Dense(hparams[HP_NUM_UNITS], activation=tf.nn.relu),
        tf.keras.layers.Dropout(hparams[HP_DROPOUT]),
        tf.keras.layers.Dense(4, activation=tf.nn.softmax)
    ])
    model.compile(
        optimizer=hparams[HP_OPTIMIZER],
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    model.fit(
        X_train,
        y_train,
        epochs=hparams[HP_EPOCHS],
        batch_size=32,
        verbose = 0,
        callbacks = [
            
        ]
    ) #hp.KerasCallback(direct, hparams), tf.keras.callbacks.TensorBoard(direct)
    _, accuracy = model.evaluate(X_test, y_test)
    #tf.keras.utils.plot_model(model, show_shapes=True, rankdir="LR")
    return accuracy

In [6]:
Permutations = []
for i in range(TuningTrials):
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = .2, shuffle=True)
    datasetX = tf.convert_to_tensor(X_train)
    datasetY = tf.convert_to_tensor(y_train)
    #I manually narrowed down these domains while I tested for specific hyperparameters due to computational and time constraints. 
    # I didn't have time to grid search every realistic combination of these values
    HP_NUM_UNITS = hp.HParam('num_units', hp.Discrete(list(range(68, 75, 1)))) 
    HP_DROPOUT = hp.HParam('dropout', hp.RealInterval(0.0, 0.0)) #hp.RealInterval(0.0, 0.1)
    HP_OPTIMIZER = hp.HParam('optimizer', hp.Discrete(['Adam'])) #, 'RMSprop', 'Sgd'
    HP_EPOCHS = hp.HParam('epochs', hp.Discrete(list(range(50, 54, 1)))) 
    METRIC_ACCURACY = 'accuracy'
    session_num = 0
    for num_units in HP_NUM_UNITS.domain.values:
        for dropout_rate in (HP_DROPOUT.domain.min_value, HP_DROPOUT.domain.max_value):
            for optimizer in HP_OPTIMIZER.domain.values:
                for epoch in HP_EPOCHS.domain.values:
                    hparams = {
                        HP_NUM_UNITS: num_units,
                        HP_DROPOUT: dropout_rate,
                        HP_OPTIMIZER: optimizer,
                        HP_EPOCHS: epoch,
                    }
                    run_name = "run-%d" %  i + "-" + str(session_num)
                    print('--- Starting trial: %s' % run_name)
                    print({h.name: hparams[h] for h in hparams})
                    Permutations.append({h.name: hparams[h] for h in hparams})
                    run(hparams=hparams, num=session_num, i=i) 
                    session_num += 1

for trial in nums:
    trial = trial / TuningTrials

for i in range(len(nums)):
    print(f"Trial {i}: {nums[i]} ")
Best = max(nums)
BestCombination = Permutations[nums.index(Best)]
print(f"Best trial: {Best}")
print(f"Best combination: {BestCombination}")


--- Starting trial: run-0-0
{'num_units': 68, 'dropout': 0.0, 'optimizer': 'Adam', 'epochs': 50}
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9795 - loss: 0.0511  
0.979522168636322
--- Starting trial: run-0-1
{'num_units': 68, 'dropout': 0.0, 'optimizer': 'Adam', 'epochs': 51}
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9659 - loss: 0.0796  
0.9658703207969666
--- Starting trial: run-0-2
{'num_units': 68, 'dropout': 0.0, 'optimizer': 'Adam', 'epochs': 52}
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9556 - loss: 0.1003  
0.9556313753128052
--- Starting trial: run-0-3
{'num_units': 68, 'dropout': 0.0, 'optimizer': 'Adam', 'epochs': 53}
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9829 - loss: 0.0514  
0.9829351305961609
--- Starting trial: run-0-4
{'num_units': 68, 'dropout': 0.0, 'optimizer': 'Adam', 'epochs': 50}
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9898 - loss: 0.0225  
0.9897611141204834
--- Starting trial: run-0-5
{'num_units': 68, 

In [10]:
def train_test_tuned_model():
    model = tf.keras.models.Sequential([
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(73, activation=tf.nn.relu),
        tf.keras.layers.Dropout(0.0),
        tf.keras.layers.Dense(73, activation=tf.nn.relu),
        tf.keras.layers.Dropout(0.0),
        tf.keras.layers.Dense(73, activation=tf.nn.relu),
        tf.keras.layers.Dropout(0.0),
        tf.keras.layers.Dense(73, activation=tf.nn.relu),
        tf.keras.layers.Dropout(0.0),
        tf.keras.layers.Dense(73, activation=tf.nn.relu),
        tf.keras.layers.Dropout(0.0),
        tf.keras.layers.Dense(4, activation=tf.nn.softmax)
    ])
    model.compile(
        optimizer= 'Adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    model.fit(
        X_train,
        y_train,
        epochs=52,
        
    )
    _, accuracy = model.evaluate(X_test, y_test)
    return accuracy


sum = 0
TestTrials = 10

for i in range(TestTrials):
    run_name = "run-%d" % (i+1)
    print('--- Starting trial: %s' % run_name)
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = .2, shuffle=True)
    datasetX = tf.convert_to_tensor(X_train)
    datasetY = tf.convert_to_tensor(y_train)
    data = train_test_tuned_model()
    sum += data
    print(data)
    
average = sum / TestTrials

print(average)


--- Starting trial: run-1
Epoch 1/52
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4812 - loss: 1.1212
Epoch 2/52
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6468 - loss: 0.8266 
Epoch 3/52
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7295 - loss: 0.6457 
Epoch 4/52
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7534 - loss: 0.5787 
Epoch 5/52
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7978 - loss: 0.5193 
Epoch 6/52
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8140 - loss: 0.4547 
Epoch 7/52
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8396 - loss: 0.4194 
Epoch 8/52
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8439 - loss: 0.3848 
Epoch 9/52
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8601 - loss: 0.3473 
Epoch 10/52
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8891 - loss: 0.2923 
Epoch 11/52
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9002 - loss: 0.2776 
Epoch 12/52
37/37 ━━━━━━━━━━━━━━━━━━━━